# Case 3 — Feature Engineering Defteri

Case 1/2, veriyi **betimledi** (profil, kalite, ilişki, davranış — rapor/notebook çıktısı, model girdisi değil). Case 3 farklı bir amaca dönüyor: anomali tespiti modeline **doğrudan girdi olacak** feature kolonları üretmek. Bu yüzden `src/services/features/` diye yeni bir paket açıldı — Case 1/2'nin betimleyici `analyzers/`'ından ayrı, üretici bir katman.

Her feature seti, `TransactionID` + yeni kolonlardan oluşan ayrı bir DataFrame olarak üretiliyor — ana `merged_transactions.parquet` değiştirilmiyor, feature'lar sonradan join edilmeye hazır bağımsız bir katman olarak kalıyor.

In [1]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "requirements.txt").exists():
            return parent
    raise RuntimeError("repo root not found — expected a requirements.txt somewhere above " + str(start))


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT

PosixPath('/home/canberk/workspace/case_study')

In [2]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from src.config import settings

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

parquet_path = settings.processed_data_path / "merged_transactions.parquet"
print(f"kaynak: {parquet_path}")

kaynak: /home/canberk/workspace/case_study/data/processed/merged_transactions.parquet


## 1. Temporal Feature'lar

`src/services/features/temporal.py`, `TransactionDT`'den (referans noktadan itibaren geçen saniye — Case 1'de belirlendiği gibi gerçek bir timestamp değil) sekiz feature üretiyor.

**Kritik varsayım — `day_of_week_proxy` / `is_weekend_proxy`:** `TransactionDT`'nin referans noktası ("gün 0") resmi olarak belgelenmemiş. Bu notebook'ta gün 0 = **2017-12-01 (Cuma)** kabul ediliyor — rastgele seçilmiş bir tarih değil, iki bağımsız araştırmayla destekleniyor:
- Bu oturumun web araştırması: en az bir kamuya açık IEEE-CIS analizi aynı tarihi referans olarak kullanıyor (Kaggle'ın discussion sayfaları JS ile render edildiği için birincil kaynak/gerekçe bağımsız doğrulanamadı — bulgunun kendisi teyit edildi, arkasındaki muhakeme değil).
- Kullanıcının araştırması: Kaggle topluluğunun ABD tatil dönemi işlem hacmi kalıplarına (Şükran Günü/Black Friday/Cyber Monday sonrası artış, Noel dönemi kayması) dayanarak aynı tarihe ulaştığını gösteriyor.

**Sonuç: yarışmayı düzenleyenler (Vesta/IEEE) tarafından resmen doğrulanmamış, ama iki bağımsız kaynaktan gerekçeli bir topluluk tahmini** — ne kesin gerçek ne tamamen rastgele. Bu yüzden kolon adı `day_of_week_proxy` (çıplak `day_of_week` değil) — isim kendisi varsayıma dayandığını işaret ediyor. Modelin asıl kullanacağı `day_of_week_sin`/`cos` ham faz sayısından hesaplanıyor, hangi güne "Pazartesi" dendiğinden bağımsız — yanlış çıksa bile döngüsel yapı bozulmaz, sadece isimlendirme kayar.

In [3]:
from src.services.features.temporal import build_temporal_features, REFERENCE_DATE

print(f"referans tarih (gün 0): {REFERENCE_DATE} ({REFERENCE_DATE.strftime('%A')})")

temporal_features = build_temporal_features(parquet_path)
print(f"şekil: {temporal_features.shape}")
temporal_features.head(10)

referans tarih (gün 0): 2017-12-01 (Friday)


şekil: (590540, 10)


,TransactionID,hour_of_day,hour_sin,hour_cos,day_of_period,day_of_week_proxy,day_of_week_sin,day_of_week_cos,is_weekend_proxy,is_low_volume_hour
0,2987000,0,0.0,1.0,1,Saturday,-0.974928,-0.222521,True,False
1,2987001,0,0.0,1.0,1,Saturday,-0.974928,-0.222521,True,False
2,2987002,0,0.0,1.0,1,Saturday,-0.974928,-0.222521,True,False
3,2987003,0,0.0,1.0,1,Saturday,-0.974928,-0.222521,True,False
4,2987004,0,0.0,1.0,1,Saturday,-0.974928,-0.222521,True,False
5,2987005,0,0.0,1.0,1,Saturday,-0.974928,-0.222521,True,False
6,2987006,0,0.0,1.0,1,Saturday,-0.974928,-0.222521,True,False
7,2987007,0,0.0,1.0,1,Saturday,-0.974928,-0.222521,True,False
8,2987008,0,0.0,1.0,1,Saturday,-0.974928,-0.222521,True,False
9,2987009,0,0.0,1.0,1,Saturday,-0.974928,-0.222521,True,False


### Doğrulama 1 — `hour_of_day`, Case 1'in saatlik tablosuyla birebir eşleşiyor mu?

In [4]:
temporal_features["hour_of_day"].value_counts().sort_index().to_frame(name="işlem sayısı")

,işlem sayısı
hour_of_day,
0,37795
1,32797
2,26732
3,20802
4,14839
5,9701
6,6007
7,3704
8,2591


Case 1, bölüm 6'daki saatlik tabloyla (saat 0: 37795, saat 18: 41639, ...) birebir aynı — beklenen, çünkü aynı `// 3600 % 24` formülü kullanılıyor.

### Doğrulama 2 — döngüsel kodlama gerçekten "yakınlık" üretiyor mu?

Saat 0 ile saat 23 arasındaki Öklid mesafesi, saat 0 ile saat 12 arasındakinden belirgin küçük olmalı (23 ile 0 takvimde komşu, 12 ile 0 en uzak nokta).

In [5]:
def cyclical_distance(hour_a, hour_b):
    row_a = temporal_features[temporal_features["hour_of_day"] == hour_a].iloc[0]
    row_b = temporal_features[temporal_features["hour_of_day"] == hour_b].iloc[0]
    return np.hypot(row_a["hour_sin"] - row_b["hour_sin"], row_a["hour_cos"] - row_b["hour_cos"])


print(f"saat 0 <-> saat 23 mesafesi: {cyclical_distance(0, 23):.4f}  (komşu saatler)")
print(f"saat 0 <-> saat 12 mesafesi: {cyclical_distance(0, 12):.4f}  (en uzak saatler, çemberin çapı)")

saat 0 <-> saat 23 mesafesi: 0.2611  (komşu saatler)
saat 0 <-> saat 12 mesafesi: 2.0000  (en uzak saatler, çemberin çapı)


0,26 vs 2,00 — komşu saatler birbirine ~8 kat daha yakın çıkıyor. Ham `hour_of_day` sayısı bunun tam tersini söylerdi (|23-0|=23, |12-0|=12 — komşu saatleri en uzak gösterirdi).

### Doğrulama 3 — `is_low_volume_hour`, Case 1'in fraud-oranı bulgusuyla tutarlı mı?

In [6]:
isfraud = pq.ParquetFile(parquet_path).read(columns=["isFraud"]).to_pandas()["isFraud"]
check = temporal_features.assign(isFraud=isfraud)

(check.groupby("is_low_volume_hour")["isFraud"].mean() * 100).to_frame(name="fraud oranı (%)")

,fraud oranı (%)
is_low_volume_hour,
False,3.270736
True,6.929591


Düşük-hacim saatlerinde (04:00-09:00) fraud oranı %6,93 — normal saatlerin (%3,27) iki katından fazla. Case 1, bölüm 6'daki bulguyla (saat 7'de %10,61, genel ortalamanın ~3 katı) tutarlı; `isFraud` burada da sadece **doğrulama amaçlı betimleyici** kullanılıyor — feature'ın kendisi tamamen `TransactionDT`'den, etikete bakılmadan üretildi.

### `day_of_week_proxy` / `is_weekend_proxy` dağılımı

In [7]:
temporal_features["day_of_week_proxy"].value_counts().to_frame(name="işlem sayısı")

,işlem sayısı
day_of_week_proxy,
Saturday,98502
Friday,86377
Tuesday,85433
Thursday,85356
Wednesday,84815
Sunday,79834
Monday,70223


In [8]:
weekend_pct = temporal_features["is_weekend_proxy"].mean() * 100
print(f"hafta sonu (proxy) işlem oranı: %{weekend_pct:.1f}")

hafta sonu (proxy) işlem oranı: %30.2


---

**Durum:** Case 3'ün ilk maddesi (temporal feature'lar) tamamlandı — 8 kolon (`hour_of_day`, `hour_sin`, `hour_cos`, `day_of_period`, `day_of_week_proxy`, `day_of_week_sin`, `day_of_week_cos`, `is_weekend_proxy`, `is_low_volume_hour`), üç ayrı doğrulamayla (Case 1 saatlik tablosu, döngüsel mesafe, fraud-oranı tutarlılığı) teyit edildi. Kalan maddeler (entity, relational, context-bazlı feature'lar) sonraki bölümlerde eklenecek.

## 2. Entity Feature'lar (`card1`)

Case brief'in istediği `user_avg_amount`, `user_transaction_count` gibi feature'lar, göründüğünden daha riskli — **veri sızıntısı** tehlikesi var.

**Sorun:** bir kartın (`card1`) toplam 100 işlem yapacağını, o kartın **ilk** işlemi sırasında gerçek hayatta henüz bilemeyiz. `user_avg_amount`'ı "bu kartın tüm işlemlerinin (geçmiş + gelecek) ortalaması" diye hesaplarsak, ilk işlemin feature'ına henüz olmamış işlemlerin bilgisini karıştırmış oluruz — tıpkı bir öğrencinin sınav notunu tahmin ederken ona dönem sonu ortalamasını ipucu olarak vermek gibi: tabii ki doğru tahmin edilir ama gerçek hayatta işe yaramaz, çünkü sınav anında dönem sonu ortalaması henüz belli değildir.

**Çözüm:** her işlem için, sadece **o işlemden önce** aynı karttan ne olmuş ona bakmak (`_so_far` sütunları).

**Ekstra karar (bu bölümün asıl amacı):** sızıntının gerçekte ne kadar fark yarattığını göstermek için, güvenli `_so_far` sütunlarının yanına, kasıtlı olarak sızıntılı bir `user_avg_amount_global` sütunu da ekliyoruz — bu sütun tüm işlemleri (geçmiş+gelecek) kullanıyor ve **hiçbir zaman modele girdi olarak kullanılmayacak**, sadece karşılaştırma için var. `src/services/features/entity.py`'deki `LEAKAGE_UNSAFE_COLUMNS` sabiti bunu kod seviyesinde işaretliyor.

In [9]:
from src.services.features.entity import build_entity_features, LEAKAGE_UNSAFE_COLUMNS

entity_features = build_entity_features(parquet_path)
print(f"şekil: {entity_features.shape}")
print(f"modelde ASLA kullanılmayacak kolonlar: {LEAKAGE_UNSAFE_COLUMNS}")
entity_features.head(10)

şekil: (590540, 7)
modelde ASLA kullanılmayacak kolonlar: {'user_avg_amount_global'}


,TransactionID,user_transaction_count_so_far,user_avg_amount_so_far,user_std_amount_so_far,user_amount_zscore,user_seconds_since_last_transaction,user_avg_amount_global
0,3230924,0,NaN,NaN,NaN,NaN,23.443000
1,3023634,0,NaN,NaN,NaN,NaN,79.666667
2,3151336,1,183.000000,NaN,NaN,2587912.0,79.666667
3,3210739,2,106.000000,108.894444,-0.725473,1766278.0,79.666667
4,3020767,0,NaN,NaN,NaN,NaN,136.400000
5,3028973,1,150.000000,NaN,NaN,179352.0,136.400000
6,3386444,2,90.000000,84.852814,-0.471405,9060311.0,136.400000
7,3504371,3,76.666667,64.291005,2.322772,3470862.0,136.400000
8,3504379,4,114.000000,91.272486,1.227095,221.0,136.400000
9,3038871,0,NaN,NaN,NaN,NaN,50.000000


### Doğrulama 1 — bir kartın ilk işleminde geçmiş bilgisi gerçekten boş mu?

In [10]:
first_transactions = entity_features[entity_features["user_transaction_count_so_far"] == 0]
print(f"{len(first_transactions)} ilk-işlem satırı")
first_transactions[["user_avg_amount_so_far", "user_std_amount_so_far", "user_amount_zscore", "user_seconds_since_last_transaction"]].isna().all()

13553 ilk-işlem satırı


user_avg_amount_so_far                 True
user_std_amount_so_far                 True
user_amount_zscore                     True
user_seconds_since_last_transaction    True
dtype: bool

Hepsi `True` — ilk işlemlerde geçmişe dayalı hiçbir feature sahte bir değerle (0 gibi) doldurulmuyor, gerçek bilgi eksikliği `NaN` olarak kalıyor.

### Doğrulama 2 — elle hesaplama ile karşılaştırma

Case 2'de tanıdığımız yüksek hacimli `card1=9500` entity'sinin ilk birkaç işlemini elle kontrol edelim.

In [11]:
raw = pq.ParquetFile(parquet_path).read(columns=["TransactionID", "card1", "TransactionAmt", "TransactionDT"]).to_pandas()
sample = raw[raw["card1"] == 9500].sort_values("TransactionDT").head(6)
check = sample.merge(entity_features, on="TransactionID")
check[["TransactionID", "TransactionAmt", "user_transaction_count_so_far", "user_avg_amount_so_far", "user_avg_amount_global"]]

,TransactionID,TransactionAmt,user_transaction_count_so_far,user_avg_amount_so_far,user_avg_amount_global
0,2987107,226.00,0,NaN,115.698119
1,2987122,80.00,1,226.000000,115.698119
2,2987161,107.95,2,153.000000,115.698119
3,2987174,107.95,3,137.983333,115.698119
4,2987225,43.00,4,130.475000,115.698119
5,2987255,54.00,5,112.980000,115.698119


In [12]:
manual_avg = sample["TransactionAmt"].iloc[:2].mean()
function_avg = check["user_avg_amount_so_far"].iloc[2]
print(f"ilk 2 işlemin ortalaması (elle hesap): {manual_avg}")
print(f"3. işlemin user_avg_amount_so_far'ı (fonksiyon): {function_avg}")
print(f"eşleşiyor mu: {manual_avg == function_avg}")

ilk 2 işlemin ortalaması (elle hesap): 153.0
3. işlemin user_avg_amount_so_far'ı (fonksiyon): 153.0
eşleşiyor mu: True


Ayrıca `user_avg_amount_global` (115,70) bu tabloda her satırda aynı — Case 2'nin `entity_behavior.py`'sinde aynı entity için bulunan `amount_mean` değeriyle birebir eşleşiyor, iki bağımsız hesaplamanın çapraz doğrulaması.

### Sızıntının gerçek boyutu — `so_far` ile `global` ne kadar farklı?

In [13]:
has_history = entity_features["user_transaction_count_so_far"] > 0
comparable = entity_features[has_history].dropna(subset=["user_avg_amount_so_far"])

gap = (comparable["user_avg_amount_global"] - comparable["user_avg_amount_so_far"]).abs()

print(f"geçmişi olan işlem sayısı: {len(comparable):,}")
print(f"ortalama mutlak fark: {gap.mean():.2f} TL")
print(f"medyan mutlak fark: {gap.median():.2f} TL")
print(f"fark 50 TL'den büyük olan işlem oranı: %{(gap > 50).mean()*100:.2f}")

geçmişi olan işlem sayısı: 576,987
ortalama mutlak fark: 17.80 TL
medyan mutlak fark: 6.07 TL
fark 50 TL'den büyük olan işlem oranı: %8.94


İşlemlerin **%8,9'unda** sızıntılı (global) ve güvenli (so_far) ortalama arasındaki fark 50 TL'yi aşıyor, ortalama fark ~17,8 TL. Bu, "sızıntılı feature kullansaydık modelimiz ne kadar yanlış bir tabloya güvenirdi" sorusuna somut bir cevap — teorik bir uyarı olarak kalmıyor, gerçek veride ölçülebiliyor.

## 3. Relational Feature'lar (`card1` × `addr1` / `DeviceInfo`)

Entity feature'lar tek bir kolonun (tutar) zaman içindeki davranışına bakıyordu. Bu bölüm, `card1`'in **bağlam kolonlarıyla** (adres, cihaz) ilişkisine bakıyor — iki yönde:

- **kart → bağlam:** "bu kart daha önce bu adresten/cihazdan işlem yapmış mıydı?" — bir kartın aniden yeni bir bölgeden/cihazdan işlem yapması, hesap ele geçirmenin (account takeover) klasik bir sinyali.
- **bağlam → kart:** "bu adres/cihaz şimdiye kadar kaç farklı karttan kullanılmış?" — bir adresin/cihazın çok sayıda farklı kart tarafından paylaşılması, "drop address" ya da cihaz taklit eden (emulator) bir fraud çetesinin klasik izi — tek başına karta bakmaktan çok daha güçlü bir sinyal.

Entity feature'larla aynı nedensel (`_so_far`) disiplin korunuyor — hiçbir feature, işlemden sonra olacakları kullanmıyor. Kod, "bağlam → kart" sayımını satır satır Python döngüsüyle değil, vektörel bir "ilk görülme" hilesiyle hesaplıyor: bağlam+zamana göre sıralanmış veride, bir satır kendi `card1`'inin o bağlam altında **ilk** görüldüğü satırsa, bunların bağlam grubu içindeki kümülatif toplamı doğrudan "o ana kadar görülen farklı kart sayısı"nı verir.

In [14]:
from src.services.features.relational import build_relational_features

relational_features = build_relational_features(parquet_path)
print(f"şekil: {relational_features.shape}")
relational_features.head(10)

şekil: (590540, 6)


,TransactionID,card_addr1_pair_count_so_far,is_new_addr1_for_card,is_new_device_for_card,addr1_distinct_cards_so_far,device_distinct_cards_so_far
0,3230924,0,False,True,NaN,6.0
1,3023634,0,True,False,152.0,NaN
2,3151336,1,False,False,353.0,NaN
3,3210739,2,False,False,419.0,NaN
4,3020767,0,True,True,512.0,421.0
5,3028973,0,True,True,497.0,417.0
6,3386444,0,True,True,218.0,2753.0
7,3504371,1,False,False,1580.0,NaN
8,3504379,2,False,False,1580.0,NaN
9,3038871,0,True,True,336.0,520.0


### Doğrulama 1 — eksiklik oranları Case 1'le tutarlı mı?

In [15]:
relational_features.isna().mean().to_frame(name="eksik oran")

,eksik oran
TransactionID,0.000000
card_addr1_pair_count_so_far,0.000000
is_new_addr1_for_card,0.000000
is_new_device_for_card,0.000000
addr1_distinct_cards_so_far,0.111264
device_distinct_cards_so_far,0.799055


`addr1_distinct_cards_so_far`'ın eksik oranı (%11,13), Case 1'in bulduğu `addr1`'in eksik oranıyla (%11,13) birebir eşleşiyor — beklenen, çünkü `addr1` boşsa bu feature'ın da tanımsız kalması gerekiyor.

### Doğrulama 2 — elle izleme

23 işlemi olan küçük bir `addr1` grubunu, art arda gelen farklı kartları elle sayarak fonksiyonun çıktısıyla karşılaştıralım.

In [16]:
raw_addr = pq.ParquetFile(parquet_path).read(columns=["TransactionID", "card1", "addr1", "TransactionDT"]).to_pandas()
counts = raw_addr["addr1"].value_counts()
mid_addr = counts[(counts >= 15) & (counts <= 25)].index[0]
sample = raw_addr[raw_addr["addr1"] == mid_addr].sort_values("TransactionDT").copy()

seen = set()
manual_prior_distinct = []
for c in sample["card1"]:
    manual_prior_distinct.append(len(seen))
    seen.add(c)
sample["manual_prior_distinct"] = manual_prior_distinct

check = sample.merge(relational_features, on="TransactionID")
print(f"addr1={mid_addr}, {len(check)} işlem")
print("elle sayım ile fonksiyon çıktısı tam eşleşiyor mu:",
      (check['manual_prior_distinct'] == check['addr1_distinct_cards_so_far']).all())
check[["TransactionID", "card1", "manual_prior_distinct", "addr1_distinct_cards_so_far"]]

addr1=483.0, 23 işlem
elle sayım ile fonksiyon çıktısı tam eşleşiyor mu: True


,TransactionID,card1,manual_prior_distinct,addr1_distinct_cards_so_far
0,3004262,16746,0,0.0
1,3004513,9633,1,1.0
2,3004516,9633,2,2.0
3,3004525,9633,2,2.0
4,3011728,16746,2,2.0
5,3012953,12823,2,2.0
6,3012956,13413,3,3.0
7,3012962,9633,4,4.0
8,3012966,10876,4,4.0
9,3012991,10876,5,5.0


### En çok "paylaşılan" adresler ve cihazlar — dikkat: `DeviceInfo`'da yanıltıcı bir sonuç var

In [17]:
raw_ctx = pq.ParquetFile(parquet_path).read(columns=["TransactionID", "addr1", "DeviceInfo"]).to_pandas()
merged = raw_ctx.merge(relational_features, on="TransactionID")

print("en çok kart paylaşan 5 addr1 (nihai sayı):")
print(merged.groupby("addr1")["addr1_distinct_cards_so_far"].max().sort_values(ascending=False).head())

print("\nen çok kart paylaşan 5 DeviceInfo (nihai sayı):")
print(merged.groupby("DeviceInfo")["device_distinct_cards_so_far"].max().sort_values(ascending=False).head())

en çok kart paylaşan 5 addr1 (nihai sayı):
addr1
299.0    2006.0
204.0    1939.0
264.0    1862.0
325.0    1645.0
330.0    1410.0
Name: addr1_distinct_cards_so_far, dtype: float64

en çok kart paylaşan 5 DeviceInfo (nihai sayı):
DeviceInfo
Windows        4846.0
iOS Device     3103.0
MacOS          2287.0
Trident/7.0    1868.0
rv:11.0         702.0
Name: device_distinct_cards_so_far, dtype: float64


**`addr1` listesi** anlamlı — belirli bölge kodları gerçekten binlerce farklı kart tarafından kullanılmış, potansiyel drop-address adayları.

**`DeviceInfo` listesi yanıltıcı.** En üstte "Windows", "iOS Device", "MacOS" gibi **jenerik işletim sistemi etiketleri** var — bunlar gerçek bir cihaz parmak izi değil, milyonlarca farklı gerçek cihazın ortak işletim sistemi adı. "Windows'u 4846 farklı kart paylaştı" demek, gerçekte hiçbir fraud sinyali taşımıyor — sadece Windows kullanan çok insan var demek. Case 1'in `DeviceInfo`'yu "yüksek kardinaliteli metin" olarak işaretlemesinin nedeni tam da bu: 1786 farklı değerin bir kısmı gerçekten ayırt edici cihaz modelleri (`SAMSUNG SM-G892A Build/NRD90M` gibi), bir kısmı ise bu jenerik OS etiketleri — `device_distinct_cards_so_far` feature'ı bu ayrımı yapmıyor, ikisini aynı kefeye koyuyor. **Pratik sonuç:** bu feature'ı kullanan bir model, jenerik OS etiketlerini önce filtrelemeli veya bu feature'ı yalnızca spesifik cihaz string'leri için hesaplamalı — şu anki haliyle üretici sinyali seyreltiyor.

### Betimleyici kontrol — `is_new_*` ile fraud oranı (yalnızca gözlem, karar verici değil)

In [18]:
isfraud = pq.ParquetFile(parquet_path).read(columns=["TransactionID", "isFraud"]).to_pandas()
check_fraud = relational_features.merge(isfraud, on="TransactionID")

print("is_new_addr1_for_card ile fraud oranı (%):")
print(check_fraud.groupby("is_new_addr1_for_card")["isFraud"].mean() * 100)
print("\nis_new_device_for_card ile fraud oranı (%):")
print(check_fraud.groupby("is_new_device_for_card")["isFraud"].mean() * 100)

is_new_addr1_for_card ile fraud oranı (%):
is_new_addr1_for_card
False    3.562691
True     2.560550
Name: isFraud, dtype: float64

is_new_device_for_card ile fraud oranı (%):
is_new_device_for_card
False    3.323004
True     7.298879
Name: isFraud, dtype: float64


İki feature **zıt yönde** davranıyor — ikisi de dürüstçe raporlanmalı, ilki sezgiye aykırı:

- **`is_new_device_for_card`**: beklenen yönde ve güçlü — yeni cihazda fraud oranı %7,30, bilinen cihazda %3,32 (iki katından fazla). Hesap ele geçirme sezgisiyle örtüşüyor.
- **`is_new_addr1_for_card`**: **beklenenin tersi** — yeni adreste fraud oranı (%2,56) bilinen adresten (%3,56) daha **düşük**. Bunun için kesin bir nedensellik iddia etmiyoruz (`isFraud` burada sadece betimleyici), ama olası bir açıklama: meşru müşteriler sık seyahat eder/taşınır (yeni adres = normal davranış), fraud işlemleri ise genelde kartın **zaten bilinen** bir teslimat adresine yönlendirilir (çalınan kart bilgisiyle, saldırganın kendi adresine değil, mağdurun sık kullandığı adrese yapılan işlemler daha az şüpheli görünsün diye). Bu, "yeni = şüpheli" gibi saf bir varsayımın her zaman doğru olmadığını gösteren iyi bir örnek — modele bu feature'ı koyarken yönünü veriden öğrenmesine izin vermek, elle bir varsayım dayatmaktan daha güvenli.

---

**Durum:** Case 3'ün 3. maddesi (relational feature'lar) tamamlandı — 5 kolon, hepsi nedensel (`_so_far`/an itibarıyla). Kalan madde (context-bazlı feature'lar) sonraki bölümde eklenecek.

## 4. Context Feature'lar (ürün tipi × saat)

Entity feature'lar "bu kullanıcı için normal mi" sorusunu, relational feature'lar "bu kart-adres/cihaz kombinasyonu tanıdık mı" sorusunu sordu. Bu son bölüm farklı bir soru soruyor: **"bu işlem, kendi bağlamındaki (aynı ürün tipi, aynı saat dilimi) diğer işlemlere göre tipik mi?"**

`src/services/features/context.py`, `entity.py`'deki expanding-mean/std mekanizmasını aynen yeniden kullanıyor — tek fark, gruplama anahtarının `card1` değil bir **bağlam segmenti** olması. İki segment seçildi (rastgele değil, önceki bölümlerdeki bulgulara dayanarak):

- **`ProductCD`** — Case 1'de bu kolonun identity kapsamasının bile ürün tipine göre çarpıcı şekilde değiştiğini bulmuştuk (bölüm 5, `W` ürününde %0 kapsama); tutar davranışının da farklı olması bekleniyor.
- **`hour_of_day`** — bu notebook'un 1. bölümünde zaten türetilmişti; Case 1'in saat-fraud oranı ilişkisi bulgusuyla doğal bir devam.

Aynı nedensel disiplin: bir segmentin ortalaması/sapması sadece o ana kadar (`_so_far`) olan işlemlerden hesaplanıyor.

In [19]:
from src.services.features.context import build_context_features

context_features = build_context_features(parquet_path)
print(f"şekil: {context_features.shape}")
context_features.head(10)

şekil: (590540, 5)


,TransactionID,productcd_avg_amount_so_far,amount_zscore_within_productcd,hour_avg_amount_so_far,amount_zscore_within_hour
0,2987000,NaN,NaN,NaN,NaN
1,2987001,68.500000,NaN,68.500000,NaN
2,2987002,48.750000,0.366979,48.750000,0.366979
3,2987003,52.166667,-0.105088,52.166667,-0.105088
4,2987004,NaN,NaN,51.625000,-0.096331
5,2987005,51.625000,-0.155611,51.300000,-0.157243
6,2987006,51.100000,7.362144,50.916667,8.240279
7,2987007,69.083333,7.689803,66.357143,8.366003
8,2987008,50.000000,NaN,110.875000,-0.726658
9,2987009,119.571429,-0.018366,100.222222,0.131602


### Doğrulama 1 — eksiklik oranları, grup sayısıyla tutarlı mı?

In [20]:
null_counts = (context_features.isna().mean() * len(context_features)).round().astype(int)
null_counts.to_frame(name="eksik satır sayısı")

,eksik satır sayısı
TransactionID,0
productcd_avg_amount_so_far,5
amount_zscore_within_productcd,10
hour_avg_amount_so_far,24
amount_zscore_within_hour,48


`productcd_avg_amount_so_far`'da ~5 eksik satır (5 `ProductCD` değerinin her birinin ilk işlemi), `hour_avg_amount_so_far`'da ~24 eksik satır (24 saatin her birinin ilk işlemi) — beklenen, çünkü sadece bir grubun **gerçekten ilk** işleminde geçmiş bilgisi olamaz.

### Doğrulama 2 — elle hesaplama

In [21]:
raw_pcd = pq.ParquetFile(parquet_path).read(columns=["TransactionID", "ProductCD", "TransactionAmt", "TransactionDT"]).to_pandas()
sample = raw_pcd[raw_pcd["ProductCD"] == "S"].sort_values("TransactionDT").head(5)
check = sample.merge(context_features, on="TransactionID")

manual_avg = sample["TransactionAmt"].iloc[:3].mean()
manual_z = (sample["TransactionAmt"].iloc[3] - manual_avg) / sample["TransactionAmt"].iloc[:3].std()

print(f"ilk 3 işlemin ortalaması (elle): {manual_avg}")
print(f"4. işlemin productcd_avg_amount_so_far'ı (fonksiyon): {check['productcd_avg_amount_so_far'].iloc[3]}")
print(f"elle z-score: {manual_z:.6f}  |  fonksiyon z-score: {check['amount_zscore_within_productcd'].iloc[3]:.6f}")

ilk 3 işlemin ortalaması (elle): 19.0
4. işlemin productcd_avg_amount_so_far'ı (fonksiyon): 19.0
elle z-score: 1.677484  |  fonksiyon z-score: 1.677484


### Bu feature neden gerekli — `ProductCD`'ler arasında tutar davranışı çok farklı

In [22]:
raw_amt = pq.ParquetFile(parquet_path).read(columns=["TransactionID", "ProductCD", "TransactionAmt"]).to_pandas()
raw_amt.groupby("ProductCD")["TransactionAmt"].mean().sort_values(ascending=False).to_frame(name="ortalama tutar")

,ortalama tutar
ProductCD,
R,168.306188
W,153.158554
H,73.170058
S,60.269487
C,42.872353


`R` ürününün ortalama tutarı (168) `C`'ninkinin (43) yaklaşık 4 katı. Yani 100 TL'lik bir işlem `C` için oldukça yüksek ama `R` için sıradan — global bir eşik ("100 TL üzeri şüpheli") bu farkı göremez, `amount_zscore_within_productcd` görebilir.

### Bağlama göre atipik işlemlerde fraud oranı yüksek mi?

In [23]:
merged = raw_amt.merge(context_features, on="TransactionID")
isfraud = pq.ParquetFile(parquet_path).read(columns=["TransactionID", "isFraud"]).to_pandas()
merged = merged.merge(isfraud, on="TransactionID")

merged["abs_zscore"] = merged["amount_zscore_within_productcd"].abs()
merged["zscore_bucket"] = pd.cut(merged["abs_zscore"], bins=[0, 1, 2, 3, float("inf")], labels=["0-1", "1-2", "2-3", "3+"])

merged.groupby("zscore_bucket", observed=True)["isFraud"].agg(fraud_rate_pct=lambda s: s.mean() * 100, count="count")

,fraud_rate_pct,count
zscore_bucket,,
0-1,3.086709,544269
1-2,8.092464,26432
2-3,9.127217,8513
3+,8.369421,11315


Net bir örüntü: `|z| < 1` (kendi ürün tipi için tipik tutar) grubunda fraud oranı %3,09 — `|z| >= 1` gruplarının hepsinde %8-9 aralığında, yaklaşık **2,5-3 kat daha yüksek**. Bu, entity feature'larda gördüğümüz aynı prensibin ("bağlama göre sapma, ham değerden daha güçlü bir sinyal") bağlam bazında da geçerli olduğunu doğruluyor.

---

**Durum:** Case 3'ün 4. ve son maddesi (context feature'lar) tamamlandı.

## Sonuç — Case 3'ün dört feature ailesi

| # | Aile | Dosya | Kolon sayısı | Ana ilke |
|---|---|---|---|---|
| 1 | Temporal | `features/temporal.py` | 8 | `TransactionDT`'den döngüsel (sin/cos) zaman sinyalleri; `day_of_week_proxy` iki bağımsız araştırmayla desteklenen ama resmi olmayan bir tahmine (2017-12-01) dayanıyor |
| 2 | Entity | `features/entity.py` | 6 | `card1` bazlı, **nedensel** (`_so_far`) tutar geçmişi; kasıtlı sızıntılı `_global` referans kolonuyla sızıntının boyutu somut ölçüldü (~%8,9 işlemde 50 TL+ fark) |
| 3 | Relational | `features/relational.py` | 5 | `card1` × `addr1`/`DeviceInfo` ilişkisi, iki yönde (yeni mi / kaç farklı kart paylaşıyor); `DeviceInfo`'nun jenerik OS etiketleriyle kirlendiği dürüstçe raporlandı |
| 4 | Context | `features/context.py` | 4 | İşlemin kendi ürün/saat segmentine göre ne kadar tipik olduğu; `entity.py` ile aynı mekanizmanın segment bazında yeniden kullanımı |

**Ortak disiplin, üç ailede de (2-4):** her feature yalnızca işlemden **önce** olanlara bakıyor — hiçbiri, üretimde bir model scoring anında bilemeyeceği bilgiyi kullanmıyor. `isFraud`, hiçbir feature'ın hesaplanmasında kullanılmadı; yalnızca doğrulama bölümlerinde betimleyici olarak (bulunan feature'ların gerçekten ayırt edici olup olmadığını göstermek için) referans alındı.

**Toplam:** 23 yeni feature kolonu (`user_avg_amount_global` hariç, o kasıtlı olarak modelden dışlanıyor), her biri `TransactionID` üzerinden ana veri setine join edilmeye hazır, dört ayrı dosyada üretiliyor.